[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn as nn
import math

In [31]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model, self.num_heads = d_model, num_heads
        # pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        q = torch.stack(self.W_q(x_q).chunk(self.num_heads, dim=-1), dim=1)
        k = torch.stack(self.W_k(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        v = torch.stack(self.W_v(x_kv).chunk(self.num_heads, dim=-1),dim=1)
        dh = self.d_model//self.num_heads
        # print(q.shape, k.shape, v.shape,)
        s = torch.einsum('ijkl,ijml->ijkm', q, k)
        # print(s.shape)
        attn = torch.softmax(s, dim=-1)
        head = torch.einsum('bhoi,bhiv->bhov', attn, v)
        # print(head.shape)
        head = head.reshape(head.shape[0], head.shape[2], head.shape[1]*head.shape[3])
        return head
        # pass  # Q from x_q, K/V from x_kv, no causal mask

In [32]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [33]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.1ms)
  ✅ [2/4] Q and KV different lengths (1.1ms)
  ✅ [3/4] No causal mask — all KV affects all Q (48.8ms)
  ✅ [4/4] Gradient flow (20.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (72.7ms total)
  Progress saved. Run status() to see your dashboard.

